<a href="https://colab.research.google.com/github/Pmskabir1234/Torch/blob/main/breast_cancer_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install opendatasets

In [80]:
import pandas as pd
import numpy as np
import opendatasets as od
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch

In [69]:
od.download("https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset")

Skipping, found downloaded files in "./breast-cancer-dataset" (use force=True to force download)


In [90]:
df = pd.read_csv("/content/breast-cancer-dataset/breast-cancer.csv")

In [91]:
df['diagnosis'].unique()

array(['M', 'B'], dtype=object)

In [92]:
df.isna().sum()

,0
id,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0


In [93]:
df = df.drop(columns='id')

In [94]:
df.head(5)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [76]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df['diagnosis'], test_size=0.2, shuffle=True)

In [95]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

type(X_test)


numpy.ndarray

In [96]:
le = LabelEncoder()
y_test = le.fit_transform(y_test)
y_train = le.fit_transform(y_train)

type(y_test)

numpy.ndarray

In [97]:
X_test = torch.from_numpy(X_test)
X_train = torch.from_numpy(X_train)
y_test = torch.from_numpy(y_test)
y_train = torch.from_numpy(y_train)

In [99]:
X_test.shape

torch.Size([114, 30])

In [103]:
lr = 0.1
Epochs = 25

In [115]:
class SimpleNN():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, requires_grad=True, dtype=torch.float64)
    self.bias = torch.zeros(1, requires_grad=True, dtype=torch.float64)

  def forward(self,X):
    z = torch.matmul(X,self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_func(self, y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    # Corrected binary cross-entropy loss
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
    return loss

In [121]:
#model instance
model = SimpleNN(X_train)

for i in range(Epochs):
  y_pred = model.forward(X_train)
  loss = model.loss_func(y_pred, y_train)
  print(f"Epoch: {i+1}  Loss: {loss.item()}")

  loss.backward()
  with torch.no_grad():
    model.weights -= lr*model.weights.grad
    model.bias -= lr*model.bias.grad

  model.weights.grad.zero_()
  model.bias.grad.zero_()


Epoch: 1  Loss: 4.096423521592182
Epoch: 2  Loss: 3.7847894656830525
Epoch: 3  Loss: 3.4827926200757524
Epoch: 4  Loss: 3.196995810917979
Epoch: 5  Loss: 2.928226464508952
Epoch: 6  Loss: 2.6801636804640863
Epoch: 7  Loss: 2.4433847089979652
Epoch: 8  Loss: 2.2268870105655916
Epoch: 9  Loss: 2.0302556401711267
Epoch: 10  Loss: 1.8522122924168178
Epoch: 11  Loss: 1.6936317736072646
Epoch: 12  Loss: 1.5536509225740631
Epoch: 13  Loss: 1.430526759643275
Epoch: 14  Loss: 1.322648987855672
Epoch: 15  Loss: 1.228521907299929
Epoch: 16  Loss: 1.1467513818762425
Epoch: 17  Loss: 1.0760363520859493
Epoch: 18  Loss: 1.015164013295469
Epoch: 19  Loss: 0.9630075716933604
Epoch: 20  Loss: 0.9185254671224719
Epoch: 21  Loss: 0.880761086300536
Epoch: 22  Loss: 0.8488422254667951
Epoch: 23  Loss: 0.8219798324530377
Epoch: 24  Loss: 0.7994658079167247
Epoch: 25  Loss: 0.7806698387005488


In [123]:
with torch.no_grad():
  y_pred = model.forward(X_test)
  y_pred = (y_pred > 0.5).float()

acc = (y_pred == y_test).float().mean()
print(acc.item())

0.5344721674919128


#Key Points

1. StandardScaler, LabelEncoder ***fit_transform*** returns a numpy array.
2. Here, no. of weights ▶ no. of features.
3. The typical dataflow to train a model ▪ Create a model ▶ Forward pass ▶ Prediction and Loss Calculation ▶ Update parameters ▶ Clear Gradients ▶ Repeat